### 1.데이터 파악

In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/combined_2024_2025.csv')
print(df.shape)

(103939, 82)


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103939 entries, 0 to 103938
Data columns (total 82 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   ID                 103939 non-null  object 
 1   WT_DOM             103939 non-null  float64
 2   BSEX               103939 non-null  int64  
 3   BAGE               103939 non-null  int64  
 4   BINC1              103939 non-null  int64  
 5   D_TRA1_SYEAR       53055 non-null   float64
 6   D_TRA1_SMONTH      53055 non-null   float64
 7   D_TRA1_1_SPOT      53055 non-null   float64
 8   D_TRA1_CASE        53055 non-null   float64
 9   D_TRA1_COST        53055 non-null   float64
 10  D_TRA1_NUM         53055 non-null   float64
 11  D_TRA1_ONE_COST    53055 non-null   float64
 12  D_TRA2_SYEAR       3014 non-null    float64
 13  D_TRA2_SMONTH      3014 non-null    float64
 14  D_TRA2_1_SPOT      3014 non-null    float64
 15  D_TRA2_CASE        3014 non-null    float64
 16  D_

In [11]:
id_cols = ['ID', 'WT_DOM', 'BSEX', 'BAGE', 'BINC1']
round_cols = [c for c in df.columns if c.startswith('D_TRA')]
region_cols = [c for c in df.columns if c.startswith('국내_A_')]
derived_cols = ['연도']

print(f"조인/가중치/통제변수: {len(id_cols)}개")
print(f"회차별 반복필드: {len(round_cols)}개")
print(f"지역 사전집계: {len(region_cols)}개")
print(f"파생변수: {len(derived_cols)}개")
print("합계:", len(id_cols)+len(round_cols)+len(region_cols)+len(derived_cols), "/ 전체:", df.shape[1])

조인/가중치/통제변수: 5개
회차별 반복필드: 42개
지역 사전집계: 34개
파생변수: 1개
합계: 82 / 전체: 82


In [12]:
print(df['연도'].value_counts())
print("ID 유일값 개수:", df['ID'].nunique(), "/ 전체 행 수:", len(df))
print("ID 중복 건수:", df['ID'].duplicated().sum())
print("(ID, 연도) 조합 중복 건수:", df.duplicated(subset=['ID', '연도']).sum())

연도
2025    52185
2024    51754
Name: count, dtype: int64
ID 유일값 개수: 103939 / 전체 행 수: 103939
ID 중복 건수: 0
(ID, 연도) 조합 중복 건수: 0


In [17]:
for n in range(1, 7):
    col = f'D_TRA{n}_CASE'
    print(f"{n}차 응답 건수: {df[col].notna().sum()}  "
          f"(2024: {df.loc[df['연도']==2024, col].notna().sum()}, "
          f"2025: {df.loc[df['연도']==2025, col].notna().sum()})")

1차 응답 건수: 53055  (2024: 26342, 2025: 26713)
2차 응답 건수: 3014  (2024: 1291, 2025: 1723)
3차 응답 건수: 210  (2024: 96, 2025: 114)
4차 응답 건수: 28  (2024: 18, 2025: 10)
5차 응답 건수: 7  (2024: 4, 2025: 3)
6차 응답 건수: 1  (2024: 0, 2025: 1)


In [14]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(1)
summary = pd.DataFrame({'결측치수': missing, '결측치비율(%)': missing_pct})
summary[summary['결측치수'] > 0].sort_values('결측치비율(%)', ascending=False)

,결측치수,결측치비율(%)
D_TRA4_SYEAR,103911,100.0
D_TRA4_SMONTH,103911,100.0
D_TRA4_CASE,103911,100.0
D_TRA4_COST,103911,100.0
D_TRA4_NUM,103911,100.0
D_TRA4_ONE_COST,103911,100.0
D_TRA5_SYEAR,103932,100.0
D_TRA5_SMONTH,103932,100.0
D_TRA5_1_SPOT,103932,100.0
D_TRA5_CASE,103932,100.0


In [15]:
for col in ['BSEX', 'BAGE', 'BINC1']:
    print(col, '→', sorted(df[col].dropna().unique()))

BSEX → [np.int64(1), np.int64(2)]
BAGE → [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
BINC1 → [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]


In [16]:
print("완전 중복행 개수:", df.duplicated().sum())

완전 중복행 개수: 0


### 1-2.데이터 타입 변경하기(float->int)

In [21]:
# int 계열로 변환: 회차별 필드 42개 전체
# float로 유지: WT_DOM, 국내_A_여행지출_* (17개)

round_int_cols = [f'D_TRA{n}_{f}' for n in range(1, 7)
                   for f in ['SYEAR','SMONTH','1_SPOT','CASE','COST','NUM','ONE_COST']]
df[round_int_cols] = df[round_int_cols].astype('Int64')

print(df[round_int_cols].dtypes.unique())                         # [Int64]
print(df['WT_DOM'].dtype, df['국내_A_여행지출_관광전체_서울'].dtype)  # float64, float64 (유지 확인)

[Int64Dtype()]
float64 float64


### 2.결측치 처리

In [ ]:
cost_num_cols = [f'D_TRA{n}_{f}' for n in range(1, 7) for f in ['COST', 'NUM', 'ONE_COST']]
date_id_cols  = [f'D_TRA{n}_{f}' for n in range(1, 7) for f in ['SYEAR', 'SMONTH', '1_SPOT', 'CASE']]

# 금액/인원 필드: 결측 -> 0으로 채우고, 최종 int64로 확정
df[cost_num_cols] = df[cost_num_cols].fillna(0).astype('int64')

# 날짜/식별 필드: 결측 그대로 유지 -> 별도 처리 없음, Int64 상태로 <NA> 유지됨
# (여행시작년, 여행시작월, 1차여행지, 응답건수) 등은 0으로 채우기보다는 결측으로 남기는 편이 좋음

# 검증
print('금액/인원 필드 dtype:', df[cost_num_cols].dtypes.unique(), '/ 결측 남은 개수:', df[cost_num_cols].isna().sum().sum())
print('날짜/식별 필드 dtype:', df[date_id_cols].dtypes.unique(), '/ 결측 남은 개수(정상):', df[date_id_cols].isna().sum().sum())

금액/인원 필드 dtype: [dtype('int64')] / 결측 남은 개수: 0
날짜/식별 필드 dtype: [Int64Dtype()] / 결측 남은 개수(정상): 2269276


### 3.이상치 처리

In [ ]:
# 로그변환 + IQR로 이상치 상/하한를 상/하한값으로 대체(윈저화)
# 표본이 너무 적은 4~6차(각 28/7/1명)는 제외(IQR이 통계적으로 의미없음, 결과 영향도 미미)

outlier_log = []
for n in [1, 2, 3]:
    has_trip = df[f'D_TRA{n}_CASE'].notna()
    for f in ['COST', 'NUM', 'ONE_COST']:
        col = f'D_TRA{n}_{f}'
        s = df.loc[has_trip, col]

        log_s = np.log1p(s)                        #로그변환으로 치우친 분포 완화
        q1, q3 = log_s.quantile(0.25), log_s.quantile(0.75)
        iqr = q3 - q1
        lo = max(0, np.expm1(q1 - 1.5 * iqr))      #원래 스케일로 역변환
        hi = np.expm1(q3 + 1.5 * iqr)

        n_low, n_high = (s < lo).sum(), (s > hi).sum()
        outlier_log.append((col, lo, hi, n_low, n_high, len(s)))

        clipped = s.clip(lower=lo, upper=hi).round().astype('int64') #범위를 벗어난 값을 상/하한값으로 조정(윈저화)
        df.loc[has_trip, col] = clipped

outlier_summary = pd.DataFrame(outlier_log, columns=['컬럼','하한','상한','하한초과','상한초과','표본수'])
print(outlier_summary)
print('\n총 winsorize된 값:', outlier_summary['하한초과'].sum() + outlier_summary['상한초과'].sum())

                컬럼            하한            상한  하한초과  상한초과    표본수
0      D_TRA1_COST  17067.046265  1.904167e+06   171   645  53055
1       D_TRA1_NUM      0.000000  1.031371e+01     0   125  53055
2  D_TRA1_ONE_COST   9621.889390  7.794115e+05   121   430  53055
3      D_TRA2_COST  15395.392081  1.247065e+06    32    46   3014
4       D_TRA2_NUM      0.000000  1.031371e+01     0     4   3014
5  D_TRA2_ONE_COST  10206.112974  4.580279e+05    43    67   3014
6      D_TRA3_COST   6064.683275  2.298161e+06     4     1    210
7       D_TRA3_NUM      0.000000  1.031371e+01     0     0    210
8  D_TRA3_ONE_COST   6238.939872  5.442099e+05     4     2    210

총 winsorize된 값: 1695


### 4.손상레코드 처리

In [24]:
#3단계에서 COST/NUM/ONE_COST를 각각 독립적으로 상하한 조정하면서 ONE_COST(=COST÷NUM)의 정합성이 깨진 걸 바로잡음
#COST, NUM은 조정된 값을 그대로 두고, ONE_COST만 그 둘로부터 다시 계산

for n in range(1, 7):
    has_trip = df[f'D_TRA{n}_CASE'].notna()
    if has_trip.sum() == 0:
        continue
    cost_col, num_col, one_col = f'D_TRA{n}_COST', f'D_TRA{n}_NUM', f'D_TRA{n}_ONE_COST'
    df.loc[has_trip, one_col] = (df.loc[has_trip, cost_col] / df.loc[has_trip, num_col]).round().astype('int64')

#검증: COST÷NUM과 ONE_COST가 다시 일치하는지 확인
for n in [1, 2, 3]:
    has_trip = df[f'D_TRA{n}_CASE'].notna()
    calc = (df.loc[has_trip, f'D_TRA{n}_COST'] / df.loc[has_trip, f'D_TRA{n}_NUM']).round()
    mismatch = (calc - df.loc[has_trip, f'D_TRA{n}_ONE_COST']).abs() > 1
    print(f"{n}차 ONE_COST 불일치 남은 개수:", mismatch.sum())

1차 ONE_COST 불일치 남은 개수: 0
2차 ONE_COST 불일치 남은 개수: 0
3차 ONE_COST 불일치 남은 개수: 0


**1차 없이 2차 이후 회차만 존재하는 272건 처리 방침**: 값 자체는 정상 범위이며, 원 설문에 회차별 진행조건(스킵로직)이 존재해 정상적으로 발생 가능한 패턴으로 판단하여 삭제하지 않고 그대로 유지함.

### 5.파생변수 생성

### 5-1.총 여행횟수, 총 지출

In [30]:
case_cols = [f'D_TRA{n}_CASE' for n in range(1, 7)]
cost_cols = [f'D_TRA{n}_COST' for n in range(1, 7)]

# 총 여행횟수: 6개 회차 중 CASE(여행 기록)가 존재하는 회차 수
df['총여행횟수'] = df[case_cols].notna().sum(axis=1)

# 총지출: 6개 회차 COST 합산 (여행 목적 무관, 응답자가 실제 지출한 국내여행 경비 전체)
df['총지출'] = df[cost_cols].sum(axis=1)

# 검증
print(df[['ID', '연도', '총여행횟수', '총지출']].head(10))
print()
print('총여행횟수 분포:')
print(df['총여행횟수'].value_counts().sort_index())
print()
print('모순 체크(여행 0회인데 지출>0):', ((df['총여행횟수']==0) & (df['총지출']>0)).sum())

                   ID    연도  총여행횟수      총지출
0  11010550271_275001  2024      0        0
1  11010550271_275003  2024      1  1020006
2  11010550271_275004  2024      1   909999
3  11010550271_275007  2024      0        0
4  11010550271_275008  2024      0        0
5  11010550271_275011  2024      1   940002
6  11010550271_275016  2024      0        0
7  11010550271_275036  2024      0        0
8  11010550271_275049  2024      0        0
9  11010550271_275062  2024      0        0

총여행횟수 분포:
총여행횟수
0    50611
1    50573
2     2557
3      171
4       21
5        5
6        1
Name: count, dtype: int64

모순 체크(여행 0회인데 지출>0): 0


### 5-2.방문 시/도 코드, 방문 시/도 명

In [31]:
sido_map = {
    11:'서울', 21:'부산', 22:'대구', 23:'인천', 24:'광주', 25:'대전', 26:'울산', 29:'세종',
    31:'경기', 32:'강원', 33:'충북', 34:'충남', 35:'전북', 36:'전남', 37:'경북', 38:'경남', 39:'제주'
}

df['방문시도코드'] = (df['D_TRA1_1_SPOT'] // 1000).astype('Int64')   # 숫자 코드 (조인용)
df['방문시도명'] = df['방문시도코드'].map(sido_map)                    # 텍스트 라벨 (읽기/시각화용)

# 검증
print(df[['D_TRA1_1_SPOT', '방문시도코드', '방문시도명']].dropna().head(5))
print()
print('매핑 실패 건수:', ((df['D_TRA1_1_SPOT'].notna()) & (df['방문시도명'].isna())).sum())
print()
print('방문시도명 분포:')
print(df['방문시도명'].value_counts())

    D_TRA1_1_SPOT  방문시도코드 방문시도명
1           32030      32    강원
2           32030      32    강원
5           38090      38    경남
11          31270      31    경기
13          34060      34    충남

매핑 실패 건수: 0

방문시도명 분포:
방문시도명
경기    7497
전남    6144
경남    5329
강원    5103
경북    5090
충남    5089
전북    3916
부산    2653
충북    2585
서울    2456
제주    2010
인천    1423
대전    1102
대구     971
울산     736
광주     517
세종     434
Name: count, dtype: int64


### 5-3.관광 포함 여부

In [33]:
case_group_bool = {1: True, 2: True, 3: False, 4: True, 5: False}

df['관광포함여부'] = df['D_TRA1_CASE'].map(case_group_bool).astype('boolean')

# 검증
print(df[['D_TRA1_CASE', '관광포함여부']].dropna().head(5))
print()
print('관광포함여부 분포:', df['관광포함여부'].value_counts(dropna=False).to_dict())
print('관광포함 비율:', df['관광포함여부'].mean())

    D_TRA1_CASE  관광포함여부
1             1    True
2             1    True
5             1    True
11            1    True
13            1    True

관광포함여부 분포: {<NA>: 50884, np.True_: 46964, np.False_: 6091}
관광포함 비율: 0.8851946093676374


### 5-4.성수기/비수기

In [34]:
conventional_peak = [1, 2, 7, 8, 12]   # 통념상 성수기
data_peak = [10, 5, 9, 1, 8]           # 데이터 기준(가중치) 상위5

df['성수기_통념'] = df['D_TRA1_SMONTH'].isin(conventional_peak).astype('boolean')
df.loc[df['D_TRA1_SMONTH'].isna(), '성수기_통념'] = pd.NA

df['성수기_데이터기준'] = df['D_TRA1_SMONTH'].isin(data_peak).astype('boolean')
df.loc[df['D_TRA1_SMONTH'].isna(), '성수기_데이터기준'] = pd.NA

# 검증
print(df[['D_TRA1_SMONTH','성수기_통념','성수기_데이터기준']].dropna().head(8))
print()
print('성수기_통념 비율:', df['성수기_통념'].mean())
print('성수기_데이터기준 비율:', df['성수기_데이터기준'].mean())

    D_TRA1_SMONTH  성수기_통념  성수기_데이터기준
1               1    True       True
2               1    True       True
5               1    True       True
11              2    True      False
13              2    True      False
16              2    True      False
17              2    True      False
18              2    True      False

성수기_통념 비율: 0.4126095561210065
성수기_데이터기준 비율: 0.4321553105268118


### 5-5.인구통계 라벨링

In [36]:
sex_map = {1: '남자', 2: '여자'}
age_map = {1: '20대 미만', 2: '20대', 3: '30대', 4: '40대', 5: '50대', 6: '60대', 7: '70세 이상'}
inc_map = {1: '100만원 미만', 2: '100~200만원 미만', 3: '200~300만원 미만', 4: '300~400만원 미만',
           5: '400~500만원 미만', 6: '500~600만원 미만', 7: '600만원 이상'}

df['성별'] = df['BSEX'].map(sex_map)
df['연령대'] = df['BAGE'].map(age_map)
df['가구소득구간'] = df['BINC1'].map(inc_map)

# 검증
print(df[['BSEX','성별','BAGE','연령대','BINC1','가구소득구간']].head(5))
for col, orig in [('성별','BSEX'), ('연령대','BAGE'), ('가구소득구간','BINC1')]:
    print(f'{col} 매핑 실패:', ((df[orig].notna()) & (df[col].isna())).sum(), '건')

   BSEX  성별  BAGE  연령대  BINC1        가구소득구간
0     1  남자     5  50대      7      600만원 이상
1     1  남자     4  40대      7      600만원 이상
2     1  남자     3  30대      5  400~500만원 미만
3     1  남자     4  40대      6  500~600만원 미만
4     2  여자     2  20대      7      600만원 이상
성별 매핑 실패: 0 건
연령대 매핑 실패: 0 건
가구소득구간 매핑 실패: 0 건
